# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "df.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "STATUS"

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(PROCESSED / "dataset_tratado.csv")
df.head()

,FLAG_OWN_CAR,FLAG_OWN_REALTY,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,STATUS,CNT_CHILDREN,AGE,YEARS_EMPLOYED,INCOME_PER_MEMBER,_F,_M,_Commercial associate,_Pensioner,_State servant,_Student,_Working,_Academic degree,_Higher education,_Incomplete higher,_Lower secondary,_Secondary / secondary special,_Married,_Not married,_Co-op apartment,_House / apartment,_Municipal apartment,_Office apartment,_Rented apartment,_With parents,_Blue-collar,_Health and Care,_High Quality staff,_Laborers,_Maintenance and Security,_Sales and Support,_Secretaries,_Unemployed,_Unknown
0,1,1,1,0,0,0,-0.559928,-0.952042,1.053965,1.531624,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True
1,1,1,0,0,0,0,-0.559928,1.313257,-0.376569,-0.564940,False,True,False,False,False,False,True,False,False,False,False,True,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False
2,0,1,0,1,1,0,-0.559928,0.790496,0.418172,1.898509,True,False,True,False,False,False,False,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False
3,0,1,0,0,0,0,-0.559928,1.574638,-0.853414,1.975132,True,False,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False
4,1,1,1,1,1,0,-0.559928,0.267734,-0.535518,0.809945,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False


## 1. Split treino/teste

`stratify` preserva a proporção das classes nos dois conjuntos.

In [3]:
# Variáveis independentes
X = df.drop(columns=[TARGET])

# Variável Alvo
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

## 2. Modelos candidatos

Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

In [8]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

logreg = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
logreg.fit(X_train, y_train)
y_pred_logreg = logreg.predict(X_test)

tree = DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=10)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

forest = RandomForestClassifier(random_state=RANDOM_STATE, max_depth=10, n_estimators=100)
forest.fit(X_train, y_train)
y_pred_forest = forest.predict(X_test)

svm = SVC(random_state=RANDOM_STATE)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)


In [11]:
modelos = {
    "KNN": Pipeline([
        ("clf", KNeighborsClassifier(n_neighbors=5)),
    ]),
    "Regressão Logística": Pipeline([
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
    "Floresta Aleatória": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')),
    ]),
    "Árvore de Decisão": Pipeline([
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
    "SVM": Pipeline([
        ("clf", SVC(random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
    "Hist Gradient Boosting": Pipeline([
        ("clf", HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE))
    ])
}

## 3. Validação cruzada

In [12]:
from sklearn.model_selection import cross_val_score

for nome, modelo in modelos.items():
    f1 = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    acc = cross_val_score(modelo, X_train, y_train, cv=5, scoring="accuracy")
    print(f'F1: {f1.mean():.4f}\t Accuracy: {acc.mean():.4f}\t{nome}')

F1: 0.0164	 Accuracy: 0.9541	KNN
F1: 0.0936	 Accuracy: 0.6119	Regressão Logística
F1: 0.0487	 Accuracy: 0.9301	Floresta Aleatória
F1: 0.0610	 Accuracy: 0.8940	Árvore de Decisão
F1: 0.0928	 Accuracy: 0.6866	SVM
F1: 0.0847	 Accuracy: 0.8609	Hist Gradient Boosting


In [6]:
from sklearn.metrics import average_precision_score, roc_auc_score

baseline_results = []

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    
    baseline_results.append({
        "Modelo": nome,
        "ROC-AUC": round(roc_auc, 4),
        "PR-AUC": round(pr_auc, 4)
    })

df_resultados = pd.DataFrame(baseline_results).sort_values(by="ROC-AUC", ascending=False)
print(df_resultados.to_string(index=False))

                Modelo  ROC-AUC  PR-AUC
Hist Gradient Boosting   0.5730  0.0587
   Regressão Logística   0.5625  0.0892
                   KNN   0.5415  0.0511
    Floresta Aleatória   0.5034  0.0473
     Árvore de Decisão   0.4974  0.0455


In [7]:
from sklearn.model_selection import StratifiedKFold, cross_validate


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['average_precision', 'roc_auc', 'f1', 'recall']

resultados = []

for nome, modelo in modelos.items():
    scores = cross_validate(modelo, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    resultados.append({
        "Modelo": nome,
        "PR-AUC (Média)": scores['test_average_precision'].mean(),
        "ROC-AUC (Média)": scores['test_roc_auc'].mean(),
        "F1-Score (Média)": scores['test_f1'].mean(),
        "Recall (Média)": scores['test_recall'].mean()
    })

df_ranking = pd.DataFrame(resultados).sort_values(by="PR-AUC (Média)", ascending=False)
print(df_ranking.to_string(index=False))

                Modelo  PR-AUC (Média)  ROC-AUC (Média)  F1-Score (Média)  Recall (Média)
   Regressão Logística        0.102107         0.560744          0.097398        0.470557
Hist Gradient Boosting        0.061302         0.544737          0.103423        0.196782
                   KNN        0.052816         0.533365          0.004396        0.002273
    Floresta Aleatória        0.050778         0.484697          0.056088        0.049770
     Árvore de Decisão        0.047570         0.510464          0.069544        0.090424


## 4. Comparação

**Leitura:** _qual modelo venceu e por qual margem? A diferença é relevante ou está dentro do ruído?_